# Day 33 — **asyncio.Queue: Agent Task Pipeline**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

A supervisor with three specialist agents is a **producer/consumer** problem, and `asyncio.Queue` is the stdlib answer. It gives you fan-out to N workers, natural load balancing (whoever's free grabs the next task), backpressure via `maxsize`, and a clean "all done" signal via `join()` — no locks anywhere, because it's all one event loop.

Today:
1. Orchestrator **puts**, three workers **consume**, `join()` waits.
2. **PriorityQueue** — urgent fraud alerts jump the queue.
3. **Backpressure** with `maxsize`.
4. **Failure handling** — `task_done()` in `finally`, and retry-by-requeue.

## 1. One queue, three workers

The orchestrator puts six tasks in. Three workers pull concurrently. `await queue.join()` blocks until every task has had `task_done()` called — that's the completion signal.

In [1]:
import asyncio, random, time

async def worker(name: str, queue: asyncio.Queue, done: list) -> None:
    """Consume tasks forever; cancelled by the orchestrator once the queue drains."""
    while True:
        task = await queue.get()
        try:
            await asyncio.sleep(task["cost"])          # stands in for an LLM call
            done.append((name, task["id"]))
            print(f"  {name} finished {task['id']}")
        finally:
            queue.task_done()                          # even on failure — see section 4

async def orchestrator() -> list:
    queue: asyncio.Queue = asyncio.Queue()
    done: list = []
    for i in range(6):
        await queue.put({"id": f"doc-{i}", "cost": 0.05})

    workers = [asyncio.create_task(worker(f"agent-{n}", queue, done)) for n in range(3)]

    t0 = time.perf_counter()
    await queue.join()                                 # wait for all 6 task_done() calls
    elapsed = time.perf_counter() - t0

    for w in workers:                                  # queue drained -> stop the workers
        w.cancel()
    await asyncio.gather(*workers, return_exceptions=True)

    print(f"\n6 tasks x 0.05s across 3 workers -> {elapsed:.2f}s (sequential would be ~0.30s)")
    return done

done = await orchestrator()
print("distribution:", {a: sum(1 for x, _ in done if x == a) for a in {a for a, _ in done}})

  agent-0 finished doc-0
  agent-1 finished doc-1
  agent-2 finished doc-2
  agent-0 finished doc-3
  agent-1 finished doc-4
  agent-2 finished doc-5

6 tasks x 0.05s across 3 workers -> 0.10s (sequential would be ~0.30s)
distribution: {'agent-0': 2, 'agent-1': 2, 'agent-2': 2}


☝ Two things to internalise. **Load balancing is free** — no scheduler decides who gets what; whichever worker is idle calls `get()` first. And `join()` only returns because every worker calls `task_done()`; forget one and your orchestrator hangs forever, which is why it belongs in a `finally`. The workers must also be explicitly cancelled — they're infinite loops, and a forgotten worker task is a leak that keeps the loop alive.

## 2. PriorityQueue — urgent tasks jump the line

A suspected-fraud alert should not wait behind fifty statement summaries. `asyncio.PriorityQueue` orders by comparison, so pair it with `@dataclass(order=True)` and mark the non-comparable fields `compare=False`.

In [2]:
from dataclasses import dataclass, field
from typing import Any

@dataclass(order=True)
class PrioritisedTask:
    priority: int                                   # 0 = urgent, 9 = background
    task_id: str = field(compare=False)
    payload: dict[str, Any] = field(compare=False, default_factory=dict)

async def priority_demo() -> None:
    q: asyncio.PriorityQueue = asyncio.PriorityQueue()
    for t in [
        PrioritisedTask(5, "statement-summary-1"),
        PrioritisedTask(5, "statement-summary-2"),
        PrioritisedTask(0, "FRAUD-ALERT-9931", {"amount": 48_000}),
        PrioritisedTask(9, "monthly-report-backfill"),
        PrioritisedTask(0, "FRAUD-ALERT-9932", {"amount": 12_500}),
    ]:
        await q.put(t)

    print("dequeue order:")
    while not q.empty():
        t = await q.get()
        print(f"  p{t.priority}  {t.task_id}")
        q.task_done()

await priority_demo()

dequeue order:
  p0  FRAUD-ALERT-9931
  p0  FRAUD-ALERT-9932
  p5  statement-summary-1
  p5  statement-summary-2
  p9  monthly-report-backfill


☝ `compare=False` on `task_id` and `payload` is essential: without it, two tasks with equal priority would be compared on their payload dicts and raise `TypeError: '<' not supported between instances of 'dict'`. Priority is *within* the queue only — a task already handed to a worker keeps running. If a fraud alert must pre-empt work in flight, you need cancellation, not a priority queue.

## 3. Backpressure — `maxsize` stops the producer running away

An unbounded queue turns a fast producer plus a slow LLM into unbounded memory growth. `maxsize` makes `put()` *await* when full — the producer is throttled by the consumers automatically.

In [3]:
async def backpressure_demo() -> None:
    q: asyncio.Queue = asyncio.Queue(maxsize=2)
    log: list[str] = []

    async def producer() -> None:
        for i in range(5):
            await q.put(i)                     # blocks once 2 are queued
            log.append(f"put {i} (qsize={q.qsize()})")

    async def slow_consumer() -> None:
        for _ in range(5):
            i = await q.get()
            await asyncio.sleep(0.03)
            log.append(f"        got {i}")
            q.task_done()

    await asyncio.gather(producer(), slow_consumer())
    print("\n".join(log))

await backpressure_demo()

put 0 (qsize=1)
put 1 (qsize=2)
put 2 (qsize=2)
        got 0
put 3 (qsize=2)
        got 1
put 4 (qsize=2)
        got 2
        got 3
        got 4


☝ The interleaving is the point: the producer races ahead for two items, then *waits* — `put()` on a full queue is a suspension point, not an error. That's backpressure, and it's the cheapest protection you have against a batch job OOM-ing a container. Rule of thumb for agent pipelines: bound every queue. `maxsize` should reflect what you can afford to hold in memory *and* re-do after a crash, since in-queue tasks are lost on restart unless you persist them (that's what Redis or SQS buy you on Day 36).

## 4. Failures — never lose a `task_done()`, retry by requeue

In [4]:
async def resilient_pipeline() -> None:
    q: asyncio.Queue = asyncio.Queue()
    dead_letter: list[dict] = []
    completed: list[str] = []

    async def call_tool(task: dict) -> str:
        if task["id"] == "doc-bad" and task["attempt"] < 2:
            raise TimeoutError("bedrock call timed out")
        return f"{task['id']} ok (attempt {task['attempt']})"

    async def worker(name: str) -> None:
        while True:
            task = await q.get()
            try:
                completed.append(await call_tool(task))
            except Exception as exc:
                task["attempt"] += 1
                if task["attempt"] < 3:
                    print(f"  {name}: {task['id']} failed ({exc}) -> requeue #{task['attempt']}")
                    await q.put(task)
                else:
                    dead_letter.append(task)
            finally:
                q.task_done()                     # ALWAYS — success, retry, or dead-letter

    for i in ["doc-1", "doc-bad", "doc-2"]:
        await q.put({"id": i, "attempt": 0})

    workers = [asyncio.create_task(worker(f"agent-{n}")) for n in range(2)]
    await q.join()
    for w in workers:
        w.cancel()
    await asyncio.gather(*workers, return_exceptions=True)

    print("\ncompleted  :", completed)
    print("dead letter:", dead_letter)

await resilient_pipeline()

  agent-0: doc-bad failed (bedrock call timed out) -> requeue #1
  agent-0: doc-bad failed (bedrock call timed out) -> requeue #2

completed  : ['doc-1 ok (attempt 0)', 'doc-2 ok (attempt 0)', 'doc-bad ok (attempt 2)']
dead letter: []


☝ Three rules doing real work here. `task_done()` lives in `finally`, so a requeued task still balances its `get()` — otherwise `join()` deadlocks on the first exception. A retry is just `await q.put(task)` with a bumped `attempt`. And the attempt cap plus a **dead-letter list** is what stops a poisoned task looping forever — the pattern every managed queue (SQS, Kafka, Celery) implements for you, and the one you must implement yourself when the queue is in-process.

## 5. Recap — the queue *is* the orchestrator

| Need | Mechanism | Trap |
|---|---|---|
| Fan out to N agents | `asyncio.Queue` + N worker tasks | workers are infinite — `cancel()` them |
| Know when all work is done | `await queue.join()` | needs every `task_done()` |
| Urgent work first | `PriorityQueue` + `@dataclass(order=True)` | mark payload fields `compare=False` |
| Cap memory | `Queue(maxsize=N)` | unbounded queue = OOM under load |
| Survive tool failures | `try/except` + requeue + attempt cap | `task_done()` in `finally`, dead-letter the rest |

A supervisor doesn't need threads or locks to drive several agents — one queue, N worker coroutines, and a `join()`. Everything you added today (priority, backpressure, retries, dead-letter) is what separates a demo pipeline from one that survives a bad afternoon in production. Today's module builds the LangGraph supervisor that this queue pattern underpins; on Day 36 the same shape moves into **Redis** so tasks survive a restart.